In [ ]:
import os
import pandas as pd
import numpy as np
import gcsfs
from datetime import datetime

In [ ]:
bucket = os.getenv('WORKSPACE_BUCKET')
bucket

In [ ]:
import hail as hl
hl.init(default_reference = "GRCh38")

## Add complementary allele columns

In [ ]:
# Function to compute the complementary allele
def complement(allele):
    complement_dict = {'A': 'T', 'C': 'G', 'T': 'A', 'G': 'C'}
    return complement_dict.get(allele, None)  # Return None if the allele is not valid

# Initialize the GCS filesystem
fs = gcsfs.GCSFileSystem()

# Define the path pattern for the files
bucket_path = 'gs://fc-secure-9afe7562-2fad-4781-ab60-03528a626c19/catalog/PGS*_hmPOS_GRCh38.txt'

# List all files that match the pattern
file_paths = fs.glob(bucket_path)

# Loop through each file path
for file_path in file_paths:
    with fs.open(file_path, 'r') as f:
        # Read the data into a Pandas DataFrame, ensuring 'hm_chr' and 'hm_pos' are strings
        df = pd.read_csv(f, sep='\t', dtype={'hm_chr': str, 'hm_pos': str})
    
    # Subset the relevant columns
    df_subset = df[['hm_chr', 'hm_pos', 'effect_allele', 'other_allele', 'effect_weight']]
    
    # Add columns for complementary alleles
    df_subset['complementary_effect_allele'] = df_subset['effect_allele'].apply(complement)
    df_subset['complementary_other_allele'] = df_subset['other_allele'].apply(complement)
    
    # Save the modified DataFrame back to GCS, replacing the original file
    with fs.open(file_path, 'w') as f_out:
        df_subset.to_csv(f_out, sep='\t', index=False)
    
    print(f"Processed and saved: {file_path}")

# WGS

## QC

In [ ]:
# write a function to streamline this
def file_QC(filename_in, filename_temp, filename_out, var_wgs):
    file = hl.import_table(filename_in, 
                               impute=True, 
                               missing="",
                               types={"hm_chr": "str",
                                      "hm_pos": "str"})
    
    file = file.annotate(variant = "chr"+file['hm_chr']+":"+file['hm_pos'] +":"+ file['other_allele'] +":"+ file['effect_allele'])    
    file = file.annotate(**hl.parse_variant(file.variant, reference_genome='GRCh38'))
    file = file.filter(hl.is_defined(file.locus))
    file = file.key_by(file.locus)
    
    # biallelic SNPs 
    file_loci_count = file.group_by(*file.key).aggregate(n_allele = hl.agg.count())
    file = file.annotate(**file_loci_count[file.locus])
    file = file.filter(file.n_allele == 1)

    # check if is_SNP
    file = file.filter(hl.is_snp(file.alleles[0], file.alleles[1])) 
    
    # Remove ambiguous SNPs
    file = file.filter(
    ~((file['effect_allele'] == 'A') & (file['other_allele'] == 'T')) &
    ~((file['effect_allele'] == 'T') & (file['other_allele'] == 'A')) &
    ~((file['effect_allele'] == 'C') & (file['other_allele'] == 'G')) &
    ~((file['effect_allele'] == 'G') & (file['other_allele'] == 'C')))
        
    file = file.checkpoint(filename_temp, overwrite=True)
    
    var_wgs = var_wgs.repartition(1000) 
    
    var_wgs = var_wgs.annotate(**file[var_wgs.locus])
    var_wgs = var_wgs.filter((~hl.is_nan(var_wgs.effect_weight)))
    
    # define mismatch
    info_match = var_wgs.filter((var_wgs.alleles[0] == var_wgs.other_allele) & (var_wgs.alleles[1] == var_wgs.effect_allele))
    info_complement = var_wgs.filter((var_wgs.alleles[0] == var_wgs.complementary_other_allele) & (var_wgs.alleles[1] == var_wgs.complementary_effect_allele))
    info_reverse = var_wgs.filter((var_wgs.alleles[0] == var_wgs.effect_allele) & (var_wgs.alleles[1] == var_wgs.other_allele))
    info_reverse_complement = var_wgs.filter((var_wgs.alleles[0] == var_wgs.complementary_effect_allele) & (var_wgs.alleles[1] == var_wgs.complementary_other_allele))
    
    # Update info_complement: Change alleles to complementary alleles
    info_complement = info_complement.annotate(
        other_allele = info_complement.complementary_other_allele,
        effect_allele = info_complement.complementary_effect_allele
    ).key_by('locus') 
    
    # Update info_reverse: Swap alleles and reverse the effect size sign
    info_reverse = info_reverse.annotate(
        effect_allele = info_reverse.other_allele,
        other_allele = info_reverse.effect_allele,
        effect_weight = -info_reverse.effect_weight  # Reverse effect size sign
    ).key_by('locus')  
    
    # Update info_reverse_complement: Swap complementary alleles and reverse effect size sign
    info_reverse_complement = info_reverse_complement.annotate(
        other_allele = info_reverse_complement.complementary_effect_allele,
        effect_allele = info_reverse_complement.complementary_other_allele,
        effect_weight = -info_reverse_complement.effect_weight  # Reverse effect size sign
    ).key_by('locus') 
    
    info_updated = info_match.union(info_complement).union(info_reverse).union(info_reverse_complement)
    
    file_QCed = info_updated.select(
                        alleles1_wgs = info_updated.alleles[0],
                        alleles2_wgs = info_updated.alleles[1],
                        beta = info_updated.effect_weight)
    
    file_QCed.export(filename_out)

In [ ]:
var_wgs = hl.read_table(f"{bucket}/WGSData/WGS_Vars_QCed.ht")
var_wgs = var_wgs.key_by("locus")

In [ ]:
print("Height_Yengo " + str(datetime.now()))
file_QC(f"{bucket}/catalog/PGS002802_hmPOS_GRCh38.txt",
        f"{bucket}/catalog/wgs_Height_Yengo_checkpoint.ht",
            f"{bucket}/catalog/wgs_Height_Yengo_QCed.tsv",
            var_wgs)

print("HDL_Kanoni " + str(datetime.now()))
file_QC(f"{bucket}/catalog/PGS002781_hmPOS_GRCh38.txt",
        f"{bucket}/catalog/wgs_HDL_Kanoni_checkpoint.ht",
            f"{bucket}/catalog/wgs_HDL_Kanoni_QCed.tsv",
            var_wgs)

print("Height_EUR " + str(datetime.now()))
file_QC(f"{bucket}/catalog/PGS005000_hmPOS_GRCh38.txt",
        f"{bucket}/catalog/wgs_Height_EUR_checkpoint.ht",
            f"{bucket}/catalog/wgs_Height_EUR_QCed.tsv",
            var_wgs)

print("Height_AFR " + str(datetime.now()))
file_QC(f"{bucket}/catalog/PGS004999_hmPOS_GRCh38.txt",
        f"{bucket}/catalog/wgs_Height_AFR_checkpoint.ht",
            f"{bucket}/catalog/wgs_Height_AFR_QCed.tsv",
            var_wgs)

print("Height_AMR " + str(datetime.now()))
file_QC(f"{bucket}/catalog/PGS004999_hmPOS_GRCh38.txt",
        f"{bucket}/catalog/wgs_Height_AMR_checkpoint.ht",
            f"{bucket}/catalog/wgs_Height_AMR_QCed.tsv",
            var_wgs)

print("HDL_AFR " + str(datetime.now()))
file_QC(f"{bucket}/catalog/PGS003773_hmPOS_GRCh38.txt",
        f"{bucket}/catalog/wgs_HDL_AFR_checkpoint.ht",
            f"{bucket}/catalog/wgs_HDL_AFR_QCed.tsv",
            var_wgs)

print("T2D_multi " + str(datetime.now()))
file_QC(f"{bucket}/catalog/PGS005242_hmPOS_GRCh38.txt",
        f"{bucket}/catalog/wgs_T2D_multi_checkpoint.ht",
            f"{bucket}/catalog/wgs_T2D_multi_QCed.tsv",
            var_wgs)

print("Breast_Cancer_multi " + str(datetime.now()))
file_QC(f"{bucket}/catalog/PGS000004_hmPOS_GRCh38.txt",
        f"{bucket}/catalog/wgs_Breast_Cancer_multi_checkpoint.ht",
            f"{bucket}/catalog/wgs_Breast_Cancer_multi_QCed.tsv",
            var_wgs)

## PRS

In [ ]:
mt_wgs = hl.read_matrix_table(f"{bucket}/WGSData/WGS_GT_QCed.mt")
mt_wgs = mt_wgs.key_rows_by("locus")

In [ ]:
mt_wgs = mt_wgs.repartition(1000)

In [ ]:
# read Sample_quant 
Sample_quant = hl.import_table(f"{bucket}/Pheno/quant_all.tsv", 
                         missing='',
                        impute=True,
                        types = {"person_id": "str"})
Sample_quant = Sample_quant.key_by("person_id")
mt_wgs_quant = mt_wgs.semi_join_cols(Sample_quant)
mt_wgs_quant = mt_wgs_quant.annotate_cols(**Sample_quant[mt_wgs_quant.s])

In [ ]:
Sample_quant_EUR = Sample_quant.filter(hl.is_defined(Sample_quant.eur_bin))
Sample_quant_AFR = Sample_quant.filter(hl.is_defined(Sample_quant.afr_bin))
Sample_quant_AMR = Sample_quant.filter(hl.is_defined(Sample_quant.amr_bin))

In [ ]:
mt_wgs_quant_EUR = mt_wgs.semi_join_cols(Sample_quant_EUR)
mt_wgs_quant_EUR = mt_wgs_quant_EUR.annotate_cols(**Sample_quant[mt_wgs_quant_EUR.s])

mt_wgs_quant_AFR = mt_wgs.semi_join_cols(Sample_quant_AFR)
mt_wgs_quant_AFR = mt_wgs_quant_AFR.annotate_cols(**Sample_quant[mt_wgs_quant_AFR.s])

mt_wgs_quant_AMR = mt_wgs.semi_join_cols(Sample_quant_AMR)
mt_wgs_quant_AMR = mt_wgs_quant_AMR.annotate_cols(**Sample_quant[mt_wgs_quant_AMR.s])

In [ ]:
#### read Sample_binary
Sample_binary = hl.import_table(f"{bucket}/YingPheno/binary_all.tsv", 
                         missing='',
                        impute=True,
                        types = {"person_id": "str"})
Sample_binary = Sample_binary.key_by("person_id")
mt_wgs_binary = mt_wgs.semi_join_cols(Sample_binary)
mt_wgs_binary = mt_wgs_binary.annotate_cols(**Sample_binary[mt_wgs_binary.s])

In [ ]:
# List of example phenotypes
phenotypes = ["Height_AFR", "HDL_AFR"]

# Loop through each phenotype and annotate it to the Matrix Table
for phenotype in phenotypes:
    print(f"Processing phenotype: {phenotype}")
    
    # Import the phenotype table
    catalog = hl.import_table(
        f'{bucket}/catalog/wgs_{phenotype}_QCed.tsv',
        impute=True,
        types = {"locus": hl.tlocus("GRCh38")}
    )
    
    catalog = catalog.key_by('locus')
    
    # Annotate rows in the Matrix Table with the phenotype data
    mt_wgs_quant_AFR = mt_wgs_quant_AFR.annotate_rows(
        **{f"{phenotype}_catalog_result": catalog[mt_wgs_quant_AFR.locus]}
    )
    

In [ ]:
# Compute PGS
phenotypes = ["Height_AFR", "HDL_AFR"]

# Annotate PGS for all phenotypes
mt_wgs_quant_AFR = mt_wgs_quant_AFR.annotate_cols(**{
    f"{phenotype}_pgs": hl.struct(
        pgs=hl.agg.sum(
            mt_wgs_quant_AFR[f"{phenotype}_catalog_result"]["beta"] * mt_wgs_quant_AFR.GT
        )
    )
    for phenotype in phenotypes
})

In [ ]:
def export_Scores(mt, pheno, mt_type, method):
    export_filename = f"{bucket}/Scores/{mt_type}/{pheno}_{method}_checkpoint.ht"
    sample_info = mt.cols().select(
        "Age", 'is_sex_Male', 'is_sex_Female', 
        "PC1", "PC2", "PC3", "PC4", "PC5", "PC6", "PC7", "PC8", 
        "PC9", "PC10", "PC11", "PC12", "PC13", "PC14", "PC15", "PC16", 
        pheno + "_pgs", pheno.split('_')[0])
    sample_info = sample_info.annotate(**sample_info[pheno + "_pgs"])
    sample_info = sample_info.drop(pheno + "_pgs")
    
    sample_info = sample_info.checkpoint(export_filename, overwrite=True) 
    print(f"Wrote sample info to {export_filename}.")

In [ ]:
print("Height_AFR " + str(datetime.now()))
export_Scores(mt_wgs_quant_AFR, "Height_AFR", "wgs", "catalog")

print("HDL_AFR " + str(datetime.now()))
export_Scores(mt_wgs_quant_AFR, "HDL_AFR", "wgs", "catalog")